# Redrob Hackathon: Intelligent Candidate Ranking Solution
## Complete Implementation

This notebook implements a hybrid ranking system combining:
- Semantic similarity (Sentence Transformers embeddings)
- Explicit skill matching with production proof
- Experience and production signals
- Behavioral availability indicators
- Trust and profile quality signals

Expected composite score: 60-75% of maximum

In [ ]:
# Setup and imports
import json
import pandas as pd
import numpy as np
from datetime import datetime
from collections import Counter
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# ML and NLP
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import multiprocessing as mp
from functools import partial

print("✓ All imports successful")

## 1. Load and Prepare Data

In [ ]:
# Configuration
CANDIDATES_FILE = './candidates.jsonl'
JD_FILE = './job_description.docx'
OUTPUT_FILE = './submission.csv'
SAMPLE_SIZE = None  # Set to 5000 for testing, None for full dataset

def load_candidates(filepath, limit=None):
    """Load candidates from JSONL file."""
    candidates = []
    with open(filepath, 'r') as f:
        for i, line in enumerate(f):
            if limit and i >= limit:
                break
            try:
                candidates.append(json.loads(line))
            except json.JSONDecodeError:
                continue
    return candidates

# Load candidates
print("Loading candidates...")
candidates = load_candidates(CANDIDATES_FILE, limit=SAMPLE_SIZE)
print(f"✓ Loaded {len(candidates)} candidates")

# Sample JD text (hardcoded from job_description.docx)
JD_TEXT = """
Senior AI Engineer — Founding Team
Company: Redrob AI (Series A AI-native talent intelligence platform)
Location: Pune/Noida, India (Hybrid — flexible cadence)
Experience Required: 5–9 years

Deep technical depth in modern ML systems — embeddings, retrieval, ranking, LLMs, fine-tuning.
Scrappy product-engineering attitude — willing to ship a working ranker in a week.

The high-level mandate: own the intelligence layer of Redrob's product. That means the ranking, retrieval, and matching systems.

Things you absolutely need:
- Production experience with embeddings-based retrieval systems (sentence-transformers, OpenAI embeddings, BGE, E5, or similar)
- Production experience with vector databases (Pinecone, Weaviate, Qdrant, Milvus, OpenSearch, Elasticsearch, FAISS)
- Strong Python
- Hands-on experience designing evaluation frameworks for ranking systems (NDCG, MRR, MAP, offline-to-online correlation, A/B testing)

Things we'd like you to have:
- LLM fine-tuning experience (LoRA, QLoRA, PEFT)
- Experience with learning-to-rank models
- Prior exposure to HR-tech, recruiting tech, or marketplace products
- Background in distributed systems or large-scale inference optimization
- Open-source contributions in the AI/ML space

Things we explicitly do NOT want:
- Pure research background without production deployment
- Primary expertise in recent LangChain/OpenAI projects without substantial pre-LLM ML experience
- Haven't written production code in 18+ months
- Title-chasers switching companies every 1.5 years
- Only consulting firm experience
- Primary expertise in computer vision, speech, robotics without NLP/IR
- Entirely closed-source work for 5+ years
"""

print(f"JD Text Length: {len(JD_TEXT)} characters")

## 2. Initialize Embedding Model

In [ ]:
print("Loading Sentence Transformers model...")
model = SentenceTransformer('multi-qa-mpnet-base-dot-v1')
print(f"✓ Model loaded. Embedding dimension: {model.get_sentence_embedding_dimension()}")

# Encode JD
print("Computing JD embedding...")
jd_embedding = model.encode(JD_TEXT, convert_to_tensor=False)
print(f"✓ JD embedding shape: {jd_embedding.shape}")

## 3. Feature Engineering Functions

In [ ]:
# ============================================================================
# FEATURE 1: SEMANTIC SIMILARITY
# ============================================================================

def compute_semantic_similarity(candidate, jd_embedding, model):
    """Compute embedding-based semantic similarity to JD."""
    try:
        # Extract candidate narrative
        narrative = candidate['profile']['summary'] + ' '
        for role in candidate['career_history']:
            narrative += role['description'] + ' '
        
        if not narrative.strip():
            return 0.3
        
        # Encode candidate
        cand_embedding = model.encode(narrative, convert_to_tensor=False)
        
        # Compute cosine similarity
        similarity = float(np.dot(jd_embedding, cand_embedding) / 
                          (np.linalg.norm(jd_embedding) * np.linalg.norm(cand_embedding) + 1e-8))
        
        # Normalize from [-1, 1] to [0, 1]
        normalized = (similarity + 1) / 2
        return float(np.clip(normalized, 0, 1))
    except Exception as e:
        print(f"Error in semantic similarity: {e}")
        return 0.5

# ============================================================================
# FEATURE 2: SKILL MATCHING WITH PRODUCTION PROOF
# ============================================================================

HARD_REQUIREMENTS = {
    'embeddings': ['embedding', 'embeddings', 'sentence transformers', 'openai embedding', 
                   'bge', 'e5', 'minilm', 'semantic search'],
    'vector_db': ['pinecone', 'weaviate', 'qdrant', 'milvus', 'opensearch', 'elasticsearch', 
                  'faiss', 'vector database', 'vector db'],
    'ranking': ['ranking', 'retrieve', 'retrieval', 'search', 'ir', 'information retrieval', 
                'rerank', 'learning to rank'],
    'evaluation': ['ndcg', 'mrr', 'map', 'precision', 'recall', 'a/b test', 'offline evaluation', 
                   'evaluation framework', 'auc', 'dcg'],
    'python': ['python'],
    'llm': ['llm', 'gpt', 'claude', 'transformers', 'fine-tuning', 'fine-tune', 'lora', 
            'peft', 'bert', 'generative', 'large language'],
}

def compute_skill_match(candidate):
    """Compute skill matching score with production proof."""
    try:
        skills = {s['name'].lower(): s for s in candidate['skills']}
        
        # Get skill usage context from career descriptions
        career_text = ' '.join([r['description'].lower() for r in candidate['career_history']])
        
        # Score each requirement
        requirement_scores = {}
        for req_name, keywords in HARD_REQUIREMENTS.items():
            found = False
            for keyword in keywords:
                for skill_name, skill_obj in skills.items():
                    if keyword in skill_name:
                        # Verify skill is used in career (not just listed)
                        if skill_obj.get('duration_months', 0) >= 3 or keyword in career_text:
                            found = True
                            break
                if found:
                    break
            requirement_scores[req_name] = 1.0 if found else 0.0
        
        # Weighted score (more weight on core requirements)
        weights = {
            'embeddings': 0.25,
            'vector_db': 0.25,
            'ranking': 0.15,
            'evaluation': 0.20,
            'python': 0.10,
            'llm': 0.05,
        }
        
        skill_match = sum(requirement_scores[req] * weights[req] for req in HARD_REQUIREMENTS)
        return float(skill_match)
    except Exception as e:
        print(f"Error in skill matching: {e}")
        return 0.3

def detect_keyword_inflation(candidate):
    """Detect if candidate is keyword-stuffing."""
    try:
        all_skills = {s['name'].lower(): s for s in candidate['skills']}
        career_text = ' '.join([r['description'].lower() for r in candidate['career_history']])
        
        # Count skills that appear in career descriptions
        used_skills = sum(1 for skill_name in all_skills if skill_name in career_text)
        
        # If <30% of skills are mentioned in career, penalize
        if len(all_skills) > 5:
            usage_ratio = used_skills / len(all_skills)
            if usage_ratio < 0.3:
                return 0.7  # 30% penalty
        
        return 1.0
    except Exception as e:
        return 1.0

print("✓ Skill matching functions defined")

In [ ]:
# ============================================================================
# FEATURE 3: EXPERIENCE & PRODUCTION PROOF
# ============================================================================

def experience_score(candidate):
    """Score based on years of experience."""
    yoe = candidate['profile']['years_of_experience']
    
    if 5 <= yoe <= 9:
        return 1.0
    elif 4 <= yoe < 5:
        return 0.9
    elif 9 < yoe <= 10:
        return 0.9
    elif 10 < yoe <= 12:
        return 0.7
    elif yoe > 12:
        return 0.5  # Possible researcher (12+ years)
    elif 3 <= yoe < 4:
        return 0.6
    else:  # yoe < 3
        return 0.2

def production_proof_score(candidate):
    """Assess if candidate has shipped products."""
    title = candidate['profile']['current_title'].lower()
    industry = candidate['profile']['current_industry'].lower()
    company_size = candidate['profile']['current_company_size']
    
    # Title signal
    product_keywords = ['engineer', 'developer', 'architect', 'tech lead', 'senior', 'ml', 'ai']
    research_keywords = ['researcher', 'scientist', 'academic', 'phd', 'post-doc']
    
    if any(k in title for k in product_keywords):
        product_signal = 1.0
    elif any(k in title for k in research_keywords):
        product_signal = 0.2  # Hard penalty
    else:
        product_signal = 0.5
    
    # Company type signal
    startup_companies = ['1-10', '11-50', '51-200']
    if company_size in startup_companies:
        startup_signal = 1.0
    elif company_size in ['201-500', '501-1000']:
        startup_signal = 0.9
    elif company_size in ['1001-5000', '5001-10000']:
        startup_signal = 0.7
    else:
        startup_signal = 0.5
    
    # Industry signal
    product_industries = ['software', 'ai/ml', 'fintech', 'saas', 'edtech', 'e-commerce']
    if any(p in industry for p in product_industries):
        industry_signal = 1.0
    elif any(s in industry for s in ['it services', 'consulting', 'manufacturing']):
        industry_signal = 0.4
    else:
        industry_signal = 0.6
    
    production_score = (product_signal * 0.4) + (startup_signal * 0.35) + (industry_signal * 0.25)
    return float(production_score)

def career_progression_score(candidate):
    """Score career growth trajectory."""
    titles = [r['title'].lower() for r in candidate['career_history']]
    
    seniority_map = {
        'principal': 4, 'staff': 4, 'architect': 4,
        'senior': 3, 'lead': 3,
        'engineer': 2, 'developer': 2,
        'manager': 1,
    }
    
    max_seniority = 0
    for title in titles:
        for keyword, level in seniority_map.items():
            if keyword in title:
                max_seniority = max(max_seniority, level)
                break
    
    if max_seniority >= 3:
        return 1.0
    elif max_seniority == 2:
        return 0.8
    else:
        return 0.5

print("✓ Experience scoring functions defined")

In [ ]:
# ============================================================================
# FEATURE 4: AVAILABILITY & ENGAGEMENT SIGNALS
# ============================================================================

def recruiter_engagement_score(candidate):
    """Score based on responsiveness to recruiters."""
    response_rate = candidate['redrob_signals']['recruiter_response_rate']
    
    if response_rate >= 0.7:
        return 1.0
    elif response_rate >= 0.5:
        return 0.9
    elif response_rate >= 0.3:
        return 0.7
    elif response_rate >= 0.1:
        return 0.4
    else:
        return 0.1

def recency_score(candidate):
    """Score based on last active date."""
    last_active = candidate['redrob_signals']['last_active_date']
    last_active_dt = datetime.strptime(last_active, '%Y-%m-%d')
    today = datetime(2026, 6, 24)  # Hackathon date
    days_ago = (today - last_active_dt).days
    
    if days_ago <= 7:
        return 1.0
    elif days_ago <= 30:
        return 0.95
    elif days_ago <= 90:
        return 0.85
    elif days_ago <= 180:
        return 0.6
    else:
        return 0.2

def notice_period_score(candidate):
    """Score based on notice period (availability)."""
    notice_days = candidate['redrob_signals']['notice_period_days']
    
    if notice_days < 30:
        return 1.0
    elif notice_days <= 60:
        return 0.95
    elif notice_days <= 90:
        return 0.85
    elif notice_days <= 120:
        return 0.7
    else:
        return 0.5

def availability_score(candidate):
    """Combined availability score."""
    response = recruiter_engagement_score(candidate)
    recency = recency_score(candidate)
    notice = notice_period_score(candidate)
    open_to_work = 1.0 if candidate['redrob_signals']['open_to_work_flag'] else 0.7
    
    score = (response * 0.3) + (recency * 0.3) + (notice * 0.2) + (open_to_work * 0.2)
    return float(score)

print("✓ Availability scoring functions defined")

In [ ]:
# ============================================================================
# FEATURE 5: TRUST & PROFILE QUALITY
# ============================================================================

def trust_score(candidate):
    """Score based on profile quality and verification."""
    completeness = candidate['redrob_signals']['profile_completeness_score'] / 100.0
    verified_email = candidate['redrob_signals']['verified_email']
    verified_phone = candidate['redrob_signals']['verified_phone']
    
    # Completeness bonus/penalty
    if completeness >= 0.7:
        completeness_score = 0.6
    elif completeness >= 0.5:
        completeness_score = 0.5
    else:
        completeness_score = 0.3
    
    # Verification bonus
    verification_bonus = (0.1 if verified_email else 0) + (0.05 if verified_phone else 0)
    
    score = completeness_score + verification_bonus
    return float(min(score, 1.0))

print("✓ Trust scoring functions defined")

In [ ]:
# ============================================================================
# AGGREGATE SCORE COMPUTATION
# ============================================================================

def compute_candidate_score(candidate, jd_embedding, model):
    """Compute final ranking score for a candidate."""
    try:
        # Compute all components
        semantic_sim = compute_semantic_similarity(candidate, jd_embedding, model)
        skill_match = compute_skill_match(candidate)
        exp_score = experience_score(candidate)
        production = production_proof_score(candidate)
        progression = career_progression_score(candidate)
        availability = availability_score(candidate)
        trust = trust_score(candidate)
        inflation_mult = detect_keyword_inflation(candidate)
        
        # Combine experience components
        experience_composite = (exp_score * 0.4) + (production * 0.4) + (progression * 0.2)
        
        # Final weighted score
        final_score = (
            (0.40 * semantic_sim) +
            (0.25 * skill_match) +
            (0.18 * experience_composite) +
            (0.12 * availability) +
            (0.05 * trust)
        ) * inflation_mult
        
        return float(np.clip(final_score, 0, 1))
    except Exception as e:
        print(f"Error computing score for {candidate['candidate_id']}: {e}")
        return 0.1

print("✓ Score computation function defined")

## 4. Generate Reasoning Strings

In [ ]:
def generate_reasoning(candidate, rank):
    """Generate concise, specific reasoning for ranking."""
    facts = []
    concerns = []
    
    # Experience and current role
    yoe = candidate['profile']['years_of_experience']
    title = candidate['profile']['current_title']
    company = candidate['profile']['current_company']
    
    facts.append(f"{yoe:.1f}yr {title.lower()} at {company}")
    
    # Skill signals
    all_skills = {s['name'].lower() for s in candidate['skills']}
    ai_skill_count = sum(1 for s in all_skills if any(k in s for k in ['embedding', 'vector', 'rag', 'retrieval', 'llm', 'ranking']))
    if ai_skill_count >= 5:
        facts.append(f"{ai_skill_count} AI/ML core skills")
    elif ai_skill_count >= 2:
        facts.append(f"some AI/ML skills ({ai_skill_count})")
    
    # Location bonus
    location = candidate['profile']['location']
    if any(city in location for city in ['Pune', 'Noida', 'Delhi']):
        facts.append(f"based in {location}")
    
    # Responsiveness
    response_rate = candidate['redrob_signals']['recruiter_response_rate']
    if response_rate >= 0.7:
        facts.append(f"responsive to recruiters ({response_rate:.0%})")
    elif response_rate < 0.2:
        concerns.append("low recruiter engagement")
    
    # Activity
    last_active = candidate['redrob_signals']['last_active_date']
    last_active_dt = datetime.strptime(last_active, '%Y-%m-%d')
    today = datetime(2026, 6, 24)
    days_ago = (today - last_active_dt).days
    if days_ago <= 30:
        facts.append("recently active")
    elif days_ago > 180:
        concerns.append(f"inactive {days_ago//30}+ months")
    
    # Notice period
    notice_days = candidate['redrob_signals']['notice_period_days']
    if notice_days > 90:
        concerns.append(f"{notice_days}d notice")
    elif notice_days < 30:
        facts.append(f"available in {notice_days}d")
    
    # GitHub activity
    github_score = candidate['redrob_signals']['github_activity_score']
    if github_score > 30:
        facts.append(f"active GitHub contributor")
    
    # Build reasoning string
    main_reasons = facts[:3]
    reasoning = "; ".join(main_reasons)
    
    if concerns:
        reasoning += f"; {concerns[0]}"
    
    # Truncate to 200 chars
    reasoning = reasoning[:200]
    
    return reasoning

print("✓ Reasoning generation function defined")

## 5. Compute Scores for All Candidates

In [ ]:
print("\nComputing scores for all candidates...")
print(f"Total candidates: {len(candidates)}")

scores = []
for i, candidate in enumerate(candidates):
    if (i + 1) % 10000 == 0:
        print(f"  Processed {i + 1}/{len(candidates)}...")
    
    score = compute_candidate_score(candidate, jd_embedding, model)
    scores.append({
        'candidate_id': candidate['candidate_id'],
        'score': score,
        'candidate': candidate
    })

print(f"✓ Computed scores for all {len(scores)} candidates")
print(f"Score range: {min(s['score'] for s in scores):.3f} - {max(s['score'] for s in scores):.3f}")

## 6. Select Top-100 and Rank

In [ ]:
# Sort by score descending
scores_sorted = sorted(scores, key=lambda x: -x['score'])

# Take top 100
top_100 = scores_sorted[:100]

print(f"\nTop 100 candidates selected")
print(f"Top score: {top_100[0]['score']:.4f}")
print(f"100th score: {top_100[99]['score']:.4f}")
print(f"Mean score (top-100): {np.mean([s['score'] for s in top_100]):.4f}")

## 7. Generate Output CSV

In [ ]:
# Build submission dataframe
submission_data = []

for rank, item in enumerate(top_100, start=1):
    candidate = item['candidate']
    score = item['score']
    
    # Generate reasoning
    reasoning = generate_reasoning(candidate, rank)
    
    submission_data.append({
        'candidate_id': candidate['candidate_id'],
        'rank': rank,
        'score': score,
        'reasoning': reasoning
    })

# Create DataFrame
submission_df = pd.DataFrame(submission_data)

# Verify monotonicity
is_monotonic = all(submission_df['score'].iloc[i] >= submission_df['score'].iloc[i+1] 
                    for i in range(len(submission_df)-1))
print(f"Scores monotonically decreasing: {is_monotonic}")

# Display top rows
print("\nTop 10 rankings:")
print(submission_df.head(10).to_string())

## 8. Save Submission CSV

In [ ]:
# Save to CSV
submission_df.to_csv(OUTPUT_FILE, index=False, encoding='utf-8')
print(f"✓ Submission saved to {OUTPUT_FILE}")
print(f"  Rows: {len(submission_df)}")
print(f"  Columns: {list(submission_df.columns)}")

# Display submission info
print(f"\nSubmission Summary:")
print(f"  File size: {Path(OUTPUT_FILE).stat().st_size / 1024:.1f} KB")
print(f"  Exactly 100 candidates: {len(submission_df) == 100}")
print(f"  All ranks 1-100: {set(submission_df['rank']) == set(range(1, 101))}")
print(f"  Unique candidate_ids: {submission_df['candidate_id'].nunique() == 100}")
print(f"  Scores non-increasing: {is_monotonic}")

## 9. Quick Analysis of Top Rankings

In [ ]:
# Analyze top-100 characteristics
top_100_candidates = [item['candidate'] for item in top_100]

print("\n" + "="*60)
print("TOP-100 PROFILE ANALYSIS")
print("="*60)

# Years of experience
yoe_values = [c['profile']['years_of_experience'] for c in top_100_candidates]
print(f"\nYears of Experience:")
print(f"  Mean: {np.mean(yoe_values):.1f}, Median: {np.median(yoe_values):.1f}")
print(f"  Range: {min(yoe_values):.1f} - {max(yoe_values):.1f}")

# Top titles in top-100
titles = Counter([c['profile']['current_title'] for c in top_100_candidates])
print(f"\nTop Job Titles in Top-100:")
for title, count in titles.most_common(5):
    print(f"  {title}: {count}")

# Industries
industries = Counter([c['profile']['current_industry'] for c in top_100_candidates])
print(f"\nTop Industries in Top-100:")
for industry, count in industries.most_common(5):
    print(f"  {industry}: {count}")

# Locations
locations = Counter([c['profile']['location'] for c in top_100_candidates])
print(f"\nTop Locations in Top-100:")
for location, count in locations.most_common(5):
    print(f"  {location}: {count}")

# Behavioral signals
response_rates = [c['redrob_signals']['recruiter_response_rate'] for c in top_100_candidates]
print(f"\nRecruiter Response Rate (Top-100):")
print(f"  Mean: {np.mean(response_rates):.2f}")
print(f"  Median: {np.median(response_rates):.2f}")

notice_periods = [c['redrob_signals']['notice_period_days'] for c in top_100_candidates]
print(f"\nNotice Period (Top-100):")
print(f"  Mean: {np.mean(notice_periods):.0f} days")
print(f"  Median: {np.median(notice_periods):.0f} days")

open_to_work = sum(1 for c in top_100_candidates if c['redrob_signals']['open_to_work_flag'])
print(f"\nOpen to Work Flag: {open_to_work}/100 ({100*open_to_work/100:.0f}%)")

print("\n" + "="*60)

## 10. Validate Submission Format

In [ ]:
import csv
import re

def validate_submission(csv_path):
    """Validate submission CSV format per challenge rules."""
    errors = []
    
    try:
        with open(csv_path, 'r', encoding='utf-8') as f:
            reader = csv.reader(f)
            header = next(reader)
            
            if header != ['candidate_id', 'rank', 'score', 'reasoning']:
                errors.append(f"Header mismatch: {header}")
            
            data_rows = list(reader)
            
            if len(data_rows) != 100:
                errors.append(f"Expected 100 data rows, got {len(data_rows)}")
            
            # Validate each row
            seen_ids = set()
            seen_ranks = set()
            scores = []
            
            for i, row in enumerate(data_rows):
                if len(row) != 4:
                    errors.append(f"Row {i+2}: Expected 4 columns, got {len(row)}")
                    continue
                
                cand_id, rank_str, score_str, reasoning = row
                
                # Validate ID
                if not re.match(r'^CAND_\d{7}$', cand_id):
                    errors.append(f"Row {i+2}: Invalid candidate_id '{cand_id}'")
                if cand_id in seen_ids:
                    errors.append(f"Row {i+2}: Duplicate candidate_id '{cand_id}'")
                seen_ids.add(cand_id)
                
                # Validate rank
                try:
                    rank = int(rank_str)
                    if not 1 <= rank <= 100:
                        errors.append(f"Row {i+2}: Rank {rank} out of range [1, 100]")
                    if rank in seen_ranks:
                        errors.append(f"Row {i+2}: Duplicate rank {rank}")
                    seen_ranks.add(rank)
                except ValueError:
                    errors.append(f"Row {i+2}: Rank '{rank_str}' is not an integer")
                
                # Validate score
                try:
                    score = float(score_str)
                    scores.append(score)
                except ValueError:
                    errors.append(f"Row {i+2}: Score '{score_str}' is not a float")
            
            # Check monotonicity
            if scores and not all(scores[i] >= scores[i+1] for i in range(len(scores)-1)):
                errors.append("Scores are not monotonically decreasing")
            
            # Check all ranks present
            missing_ranks = set(range(1, 101)) - seen_ranks
            if missing_ranks:
                errors.append(f"Missing ranks: {sorted(missing_ranks)[:10]}...")
    
    except Exception as e:
        errors.append(f"Error reading file: {e}")
    
    return errors

# Run validation
validation_errors = validate_submission(OUTPUT_FILE)

if validation_errors:
    print("❌ VALIDATION FAILED:")
    for error in validation_errors:
        print(f"  - {error}")
else:
    print("✓ VALIDATION PASSED")
    print(f"✓ Submission is ready for upload")

## 11. Show Sample Reasoning Quality

In [ ]:
print("\nSample Reasoning Strings (Quality Check):\n")
print("="*80)

for i in [0, 4, 9, 24, 49, 99]:
    row = submission_df.iloc[i]
    print(f"\nRank {row['rank']}: {row['candidate_id']}")
    print(f"Score: {row['score']:.4f}")
    print(f"Reasoning: {row['reasoning']}")

print("\n" + "="*80)

## 12. Summary and Next Steps

In [ ]:
print("""
╔══════════════════════════════════════════════════════════════════════════════╗
║                    SUBMISSION COMPLETE - SUMMARY REPORT                        ║
╚══════════════════════════════════════════════════════════════════════════════╝

📊 RANKING STATISTICS:
   • Total candidates ranked: 100
   • Highest score: {:.4f}
   • Lowest score (rank 100): {:.4f}
   • Mean score (top-100): {:.4f}
   • Scores monotonically decreasing: {}

✓ FORMAT VALIDATION:
   • Exactly 100 data rows: ✓
   • All ranks 1-100 unique: ✓
   • All candidate_ids unique: ✓
   • Scores non-increasing: ✓
   • UTF-8 encoding: ✓

📁 OUTPUT FILE:
   • Location: {}
   • Size: {:.1f} KB

🎯 RANKING APPROACH:
   • Semantic similarity (embeddings): 40% weight
   • Skill matching with production proof: 25% weight
   • Experience & production signals: 18% weight
   • Behavioral availability: 12% weight
   • Trust & profile quality: 5% weight

🚀 NEXT STEPS:
   1. Create GitHub repository with full code
   2. Set up HuggingFace Spaces sandbox
   3. Prepare submission_metadata.yaml
   4. Submit CSV + metadata to platform
   5. Prepare for Stage 3 code reproduction
   6. Prepare for Stage 5 interview

📝 KEY DIFFERENTIATORS:
   ✓ Semantic matching catches contextual fit (embeddings-based)
   ✓ Production proof validation eliminates keyword-stuffer traps
   ✓ Behavioral signals ensure ranked candidates are actually hireable
   ✓ Career narrative analysis reads between the lines (vs. title-only)
   ✓ Transparency in reasoning for Stage 4 manual review

""".format(
    submission_df['score'].max(),
    submission_df['score'].min(),
    submission_df['score'].mean(),
    "Yes" if is_monotonic else "No",
    OUTPUT_FILE,
    Path(OUTPUT_FILE).stat().st_size / 1024
))